In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import sys
sys.path.append(str(Path("../src").resolve()))

from config import (
    RAW_ROOT,
    PROCESSED_GROUP2,
    NOTCH_FREQ,
    LOW_FREQ,
    HIGH_FREQ,
    WINDOW_SIZE_SEC,
    STEP_SEC,
    BAD_CHANNEL_Z_THRESH
)

from utils import (
    load_raw_edf,
    prepare_raw,
    apply_filters,
    parse_chbmit_summary,
    detect_bad_channels,
    reorder_channels,
    zscore_normalize,
    create_windows_and_labels,
    save_processed_data,
    find_common_channels_in_order
)

# =========================================================
# TEST ONLY ONE GROUP 2 PATIENT
# =========================================================
patient_name = "chb14"

print("=" * 80)
print(f"TESTING GROUP 2 PIPELINE ON PATIENT: {patient_name}")
print("=" * 80)

patient_folder = RAW_ROOT / patient_name
summary_path = patient_folder / f"{patient_name}-summary.txt"

output_folder = PROCESSED_GROUP2 / patient_name
output_folder.mkdir(parents=True, exist_ok=True)

edf_files = sorted(patient_folder.glob("*.edf"))

print(f"Found {len(edf_files)} EDF files for {patient_name}")

if len(edf_files) == 0:
    print("No EDF files found. Stop here.")
else:
    # Read summary once
    summary_dict = parse_chbmit_summary(summary_path)

    # Get common channels across all files of this patient
    expected_channels = find_common_channels_in_order(edf_files)

    print("\nCommon channels across all files:")
    print(expected_channels)
    print(f"Number of common channels: {len(expected_channels)}")

    # summary rows
    patient_summary_rows = []

    # Process all files of chb14
    for file_path in edf_files:
        file_name = file_path.name
        seizure_intervals = summary_dict.get(file_name, [])

        print("\n" + "-" * 60)
        print(f"Processing file: {file_name}")
        print(f"Seizure intervals: {seizure_intervals}")

        # Load
        raw = load_raw_edf(file_path)
        print("Loaded raw file successfully")

        # Keep EEG channels only
        raw = prepare_raw(raw)

        print("EEG channels kept before reorder:")
        print(raw.ch_names)
        print(f"Number of EEG channels before reorder: {len(raw.ch_names)}")

        # Reorder / keep only common channels
        raw = reorder_channels(raw, expected_channels)

        print("EEG channels after reorder/common-channel selection:")
        print(raw.ch_names)
        print(f"Number of EEG channels after reorder: {len(raw.ch_names)}")

        # Sampling rate
        print(f"Original sampling rate: {raw.info['sfreq']} Hz")

        # Filter
        raw = apply_filters(
            raw,
            notch_freq=NOTCH_FREQ,
            l_freq=LOW_FREQ,
            h_freq=HIGH_FREQ
        )

        print(f"Filtering done: notch {NOTCH_FREQ} Hz + band-pass {LOW_FREQ}-{HIGH_FREQ} Hz")

        # Bad channels
        bad_channels, channel_std = detect_bad_channels(raw, z_thresh=BAD_CHANNEL_Z_THRESH)
        raw.info["bads"] = bad_channels
        print(f"Suspicious bad channels: {bad_channels}")

        # NumPy conversion
        data_before_norm = raw.get_data()
        sfreq = raw.info["sfreq"]

        print(f"Data shape before normalization: {data_before_norm.shape}")
        print(f"Sampling frequency used: {sfreq}")

        # Normalize
        data_after_norm = zscore_normalize(data_before_norm.copy())
        data = data_after_norm
        print("Normalization done")

        # Window + label
        X, y = create_windows_and_labels(
            data=data,
            sfreq=sfreq,
            window_size_sec=WINDOW_SIZE_SEC,
            step_sec=STEP_SEC,
            seizure_intervals=seizure_intervals
        )

        print(f"Windowed data shape: {X.shape}")
        print(f"Labels shape: {y.shape}")
        print(f"Number of seizure windows: {np.sum(y)}")
        print(f"Number of non-seizure windows: {len(y) - np.sum(y)}")

        # Save
        save_path = output_folder / f"{Path(file_name).stem}_processed.npz"
        save_processed_data(save_path, X, y)

        print(f"Preprocessing finished successfully for {file_name}")

        # summary row
        row = {
            "patient_name": patient_name,
            "file_name": file_name,
            "n_channels_after_common_selection": len(raw.ch_names),
            "sampling_rate": raw.info["sfreq"],
            "duration_sec": raw.n_times / raw.info["sfreq"],
            "n_seizure_intervals": len(seizure_intervals),
            "n_bad_channels": len(bad_channels),
            "n_windows": len(y),
            "n_seizure_windows": int(np.sum(y == 1)),
            "n_nonseizure_windows": int(np.sum(y == 0))
        }
        patient_summary_rows.append(row)

    # Save test summary for chb14
    df_patient_summary = pd.DataFrame(patient_summary_rows)
    patient_summary_save_path = output_folder / f"{patient_name}_processing_summary.csv"
    df_patient_summary.to_csv(patient_summary_save_path, index=False)

    print("\n" + "=" * 80)
    print(f"TEST FINISHED SUCCESSFULLY FOR {patient_name}")
    print(f"Summary saved to: {patient_summary_save_path}")
    print(df_patient_summary.head())

TESTING GROUP 2 PIPELINE ON PATIENT: chb14
Found 26 EDF files for chb14


C:\Users\MSI\Desktop\EEG_FYP\src\utils.py:20: RuntimeWarning: Channel names are not unique, found duplicates for: {'-', 'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)
C:\Users\MSI\Desktop\EEG_FYP\src\utils.py:20: RuntimeWarning: Scaling factor is not defined in following channels:
--0, --1, --2, --3, --4
  raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)
C:\Users\MSI\Desktop\EEG_FYP\src\utils.py:20: RuntimeWarning: Channel names are not unique, found duplicates for: {'-', 'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)
C:\Users\MSI\Desktop\EEG_FYP\src\utils.py:20: RuntimeWarning: Scaling factor is not defined in following channels:
--0, --1, --2, --3, --4
  raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)
C:\Users\MSI\Desktop\EEG_FYP\src\utils.py:20: RuntimeWarning: Channel names are not unique, found duplicates


Common channels across all files:
['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', '--0', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', '--1', 'FZ-CZ', 'CZ-PZ', '--2', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', '--3', 'FP2-F8', 'F8-T8', 'T8-P8-0', 'P8-O2', '--4', 'P7-T7', 'T7-FT9', 'FT9-FT10', 'FT10-T8', 'T8-P8-1']
Number of common channels: 28

------------------------------------------------------------
Processing file: chb14_01.edf
Seizure intervals: []


C:\Users\MSI\Desktop\EEG_FYP\src\utils.py:20: RuntimeWarning: Channel names are not unique, found duplicates for: {'-', 'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)
C:\Users\MSI\Desktop\EEG_FYP\src\utils.py:20: RuntimeWarning: Scaling factor is not defined in following channels:
--0, --1, --2, --3, --4
  raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)


Loaded raw file successfully
EEG channels kept before reorder:
['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', '--0', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', '--1', 'FZ-CZ', 'CZ-PZ', '--2', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', '--3', 'FP2-F8', 'F8-T8', 'T8-P8-0', 'P8-O2', '--4', 'P7-T7', 'T7-FT9', 'FT9-FT10', 'FT10-T8', 'T8-P8-1']
Number of EEG channels before reorder: 28
EEG channels after reorder/common-channel selection:
['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', '--0', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', '--1', 'FZ-CZ', 'CZ-PZ', '--2', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', '--3', 'FP2-F8', 'F8-T8', 'T8-P8-0', 'P8-O2', '--4', 'P7-T7', 'T7-FT9', 'FT9-FT10', 'FT10-T8', 'T8-P8-1']
Number of EEG channels after reorder: 28
Original sampling rate: 256.0 Hz
Filtering done: notch 60 Hz + band-pass 0.5-40 Hz

Channel STD values:
FP1-F7: 0.000059
F7-T7: 0.000061
T7-P7: 0.000055
P7-O1: 0.000047
--0: 0.000000
FP1-F3: 0.000107
F3-C3: 0.000064
C3-P3: 0.000065
P3-O1: 0.000050
--1: 0.000000
FZ-CZ: 0.000063
CZ-PZ: 0.

...

Loaded raw file successfully
EEG channels kept before reorder:
['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', '--0', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', '--1', 'FZ-CZ', 'CZ-PZ', '--2', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', '--3', 'FP2-F8', 'F8-T8', 'T8-P8-0', 'P8-O2', '--4', 'P7-T7', 'T7-FT9', 'FT9-FT10', 'FT10-T8', 'T8-P8-1']
Number of EEG channels before reorder: 28
EEG channels after reorder/common-channel selection:
['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', '--0', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', '--1', 'FZ-CZ', 'CZ-PZ', '--2', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', '--3', 'FP2-F8', 'F8-T8', 'T8-P8-0', 'P8-O2', '--4', 'P7-T7', 'T7-FT9', 'FT9-FT10', 'FT10-T8', 'T8-P8-1']
Number of EEG channels after reorder: 28
Original sampling rate: 256.0 Hz
Filtering done: notch 60 Hz + band-pass 0.5-40 Hz

Channel STD values:
FP1-F7: 0.000030
F7-T7: 0.000044
T7-P7: 0.000055
P7-O1: 0.000055
--0: 0.000000
FP1-F3: 0.000067
F3-C3: 0.000066
C3-P3: 0.000062
P3-O1: 0.000064
--1: 0.000000
FZ-CZ: 0.000063
CZ-PZ: 0.

C:\Users\MSI\Desktop\EEG_FYP\src\utils.py:20: RuntimeWarning: Channel names are not unique, found duplicates for: {'-', 'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)
C:\Users\MSI\Desktop\EEG_FYP\src\utils.py:20: RuntimeWarning: Scaling factor is not defined in following channels:
--0, --1, --2, --3, --4
  raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)


Loaded raw file successfully
EEG channels kept before reorder:
['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', '--0', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', '--1', 'FZ-CZ', 'CZ-PZ', '--2', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', '--3', 'FP2-F8', 'F8-T8', 'T8-P8-0', 'P8-O2', '--4', 'P7-T7', 'T7-FT9', 'FT9-FT10', 'FT10-T8', 'T8-P8-1']
Number of EEG channels before reorder: 28
EEG channels after reorder/common-channel selection:
['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', '--0', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', '--1', 'FZ-CZ', 'CZ-PZ', '--2', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', '--3', 'FP2-F8', 'F8-T8', 'T8-P8-0', 'P8-O2', '--4', 'P7-T7', 'T7-FT9', 'FT9-FT10', 'FT10-T8', 'T8-P8-1']
Number of EEG channels after reorder: 28
Original sampling rate: 256.0 Hz
Filtering done: notch 60 Hz + band-pass 0.5-40 Hz

Channel STD values:
FP1-F7: 0.000030
F7-T7: 0.000045
T7-P7: 0.000057
P7-O1: 0.000047
--0: 0.000000
FP1-F3: 0.000069
F3-C3: 0.000067
C3-P3: 0.000063
P3-O1: 0.000059
--1: 0.000000
FZ-CZ: 0.000061
CZ-PZ: 0.

C:\Users\MSI\Desktop\EEG_FYP\src\utils.py:20: RuntimeWarning: Channel names are not unique, found duplicates for: {'-', 'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)
C:\Users\MSI\Desktop\EEG_FYP\src\utils.py:20: RuntimeWarning: Scaling factor is not defined in following channels:
--0, --1, --2, --3, --4
  raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)


Loaded raw file successfully
EEG channels kept before reorder:
['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', '--0', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', '--1', 'FZ-CZ', 'CZ-PZ', '--2', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', '--3', 'FP2-F8', 'F8-T8', 'T8-P8-0', 'P8-O2', '--4', 'P7-T7', 'T7-FT9', 'FT9-FT10', 'FT10-T8', 'T8-P8-1']
Number of EEG channels before reorder: 28
EEG channels after reorder/common-channel selection:
['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', '--0', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', '--1', 'FZ-CZ', 'CZ-PZ', '--2', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', '--3', 'FP2-F8', 'F8-T8', 'T8-P8-0', 'P8-O2', '--4', 'P7-T7', 'T7-FT9', 'FT9-FT10', 'FT10-T8', 'T8-P8-1']
Number of EEG channels after reorder: 28
Original sampling rate: 256.0 Hz
Filtering done: notch 60 Hz + band-pass 0.5-40 Hz

Channel STD values:
FP1-F7: 0.000039
F7-T7: 0.000059
T7-P7: 0.000059
P7-O1: 0.000049
--0: 0.000000
FP1-F3: 0.000081
F3-C3: 0.000063
C3-P3: 0.000061
P3-O1: 0.000056
--1: 0.000000
FZ-CZ: 0.000064
CZ-PZ: 0.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

sys.path.append(str(Path("../src").resolve()))

from config import RAW_ROOT, GROUP2_PATIENTS, CHANNEL_ORDER
from utils import load_raw_edf, prepare_raw
group2_summary_rows = []

for patient_name in GROUP2_PATIENTS:
    patient_folder = RAW_ROOT / patient_name
    edf_files = sorted(patient_folder.glob("*.edf"))

    if len(edf_files) == 0:
        print(f"No EDF files found for {patient_name}")
        continue

    reference_file = edf_files[0]

    raw = load_raw_edf(reference_file)
    raw = prepare_raw(raw)

    row = {
        "patient": patient_name,
        "n_edf_files": len(edf_files),
        "reference_file": reference_file.name,
        "reference_sfreq": raw.info["sfreq"],
        "n_channels": len(raw.ch_names),
        "channels": raw.ch_names
    }

    group2_summary_rows.append(row)

df_group2_reference = pd.DataFrame(group2_summary_rows)

print(df_group2_reference[["patient", "n_edf_files", "reference_file", "reference_sfreq", "n_channels"]])

for _, row in df_group2_reference.iterrows():
    print("\n" + "=" * 70)
    print(f"Patient: {row['patient']}")
    print(f"Reference file: {row['reference_file']}")
    print(f"Channels ({row['n_channels']}):")
    print(row["channels"])

    group2_patient_channel_sets = {}
group2_all_file_channel_sets = []

for patient_name in GROUP2_PATIENTS:
    patient_folder = RAW_ROOT / patient_name
    edf_files = sorted(patient_folder.glob("*.edf"))

    patient_common = None

    for file_path in edf_files:
        raw = load_raw_edf(file_path)
        raw = prepare_raw(raw)
        current_channels = set(raw.ch_names)

        group2_all_file_channel_sets.append(current_channels)

        if patient_common is None:
            patient_common = current_channels
        else:
            patient_common = patient_common.intersection(current_channels)

    group2_patient_channel_sets[patient_name] = patient_common

for patient_name, ch_set in group2_patient_channel_sets.items():
    print("\n" + "=" * 70)
    print(f"Patient: {patient_name}")
    print(f"Common channels across all files of this patient: {len(ch_set)}")
    print(sorted(ch_set))

    group2_global_common = None

for patient_name, ch_set in group2_patient_channel_sets.items():
    if group2_global_common is None:
        group2_global_common = set(ch_set)
    else:
        group2_global_common = group2_global_common.intersection(ch_set)

print("\n" + "=" * 70)
print("Common channels across ALL Group 2 patients:")
print(f"Count: {len(group2_global_common)}")
print(sorted(group2_global_common))

#if the len of the group 1 common chanel is less than 19 then we wont take it 
#but in this case is not after the checks 



C:\Users\MSI\Desktop\EEG_FYP\src\utils.py:20: RuntimeWarning: Channel names are not unique, found duplicates for: {'-', 'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)
C:\Users\MSI\Desktop\EEG_FYP\src\utils.py:20: RuntimeWarning: Scaling factor is not defined in following channels:
--0, --1, --2, --3, --4
  raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)
C:\Users\MSI\Desktop\EEG_FYP\src\utils.py:20: RuntimeWarning: Channel names are not unique, found duplicates for: {'.', 'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)
C:\Users\MSI\Desktop\EEG_FYP\src\utils.py:20: RuntimeWarning: Channel names are not unique, found duplicates for: {'-', 'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)
C:\Users\MSI\Desktop\EEG_FYP\src\utils.py:20: RuntimeWarning: Scaling factor

  patient  n_edf_files reference_file  reference_sfreq  n_channels
0   chb14           26   chb14_01.edf            256.0          28
1   chb20           29   chb20_01.edf            256.0          28
2   chb21           33   chb21_01.edf            256.0          28
3   chb22           31   chb22_01.edf            256.0          28

Patient: chb14
Reference file: chb14_01.edf
Channels (28):
['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', '--0', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', '--1', 'FZ-CZ', 'CZ-PZ', '--2', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', '--3', 'FP2-F8', 'F8-T8', 'T8-P8-0', 'P8-O2', '--4', 'P7-T7', 'T7-FT9', 'FT9-FT10', 'FT10-T8', 'T8-P8-1']

Patient: chb20
Reference file: chb20_01.edf
Channels (28):
['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', '.-0', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', '.-1', 'FZ-CZ', 'CZ-PZ', '.-2', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', '.-3', 'FP2-F8', 'F8-T8', 'T8-P8-0', 'P8-O2', '.-4', 'P7-T7', 'T7-FT9', 'FT9-FT10', 'FT10-T8', 'T8-P8-1']

Patient: chb21
Reference file:

C:\Users\MSI\Desktop\EEG_FYP\src\utils.py:20: RuntimeWarning: Channel names are not unique, found duplicates for: {'-', 'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)
C:\Users\MSI\Desktop\EEG_FYP\src\utils.py:20: RuntimeWarning: Scaling factor is not defined in following channels:
--0, --1, --2, --3, --4
  raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)
C:\Users\MSI\Desktop\EEG_FYP\src\utils.py:20: RuntimeWarning: Channel names are not unique, found duplicates for: {'-', 'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)
C:\Users\MSI\Desktop\EEG_FYP\src\utils.py:20: RuntimeWarning: Scaling factor is not defined in following channels:
--0, --1, --2, --3, --4
  raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)
C:\Users\MSI\Desktop\EEG_FYP\src\utils.py:20: RuntimeWarning: Channel names are not unique, found duplicates


Patient: chb14
Common channels across all files of this patient: 28
['--0', '--1', '--2', '--3', '--4', 'C3-P3', 'C4-P4', 'CZ-PZ', 'F3-C3', 'F4-C4', 'F7-T7', 'F8-T8', 'FP1-F3', 'FP1-F7', 'FP2-F4', 'FP2-F8', 'FT10-T8', 'FT9-FT10', 'FZ-CZ', 'P3-O1', 'P4-O2', 'P7-O1', 'P7-T7', 'P8-O2', 'T7-FT9', 'T7-P7', 'T8-P8-0', 'T8-P8-1']

Patient: chb20
Common channels across all files of this patient: 28
['.-0', '.-1', '.-2', '.-3', '.-4', 'C3-P3', 'C4-P4', 'CZ-PZ', 'F3-C3', 'F4-C4', 'F7-T7', 'F8-T8', 'FP1-F3', 'FP1-F7', 'FP2-F4', 'FP2-F8', 'FT10-T8', 'FT9-FT10', 'FZ-CZ', 'P3-O1', 'P4-O2', 'P7-O1', 'P7-T7', 'P8-O2', 'T7-FT9', 'T7-P7', 'T8-P8-0', 'T8-P8-1']

Patient: chb21
Common channels across all files of this patient: 28
['--0', '--1', '--2', '--3', '--4', 'C3-P3', 'C4-P4', 'CZ-PZ', 'F3-C3', 'F4-C4', 'F7-T7', 'F8-T8', 'FP1-F3', 'FP1-F7', 'FP2-F4', 'FP2-F8', 'FT10-T8', 'FT9-FT10', 'FZ-CZ', 'P3-O1', 'P4-O2', 'P7-O1', 'P7-T7', 'P8-O2', 'T7-FT9', 'T7-P7', 'T8-P8-0', 'T8-P8-1']

Patient: chb22
Common

In [1]:
print("Final decision candidate:")
print(len(group2_global_common), group2_global_common)

# its not because we have 23 channel 

if save_path.exists():
        print("\n" + "-" * 60)
        print(f"Skipping preprocessing for {file_name} because it already exists.")

        saved_data = np.load(save_path)
        X = saved_data["X"]
        y = saved_data["y"]

        raw = load_raw_edf(file_path)
        raw = prepare_raw(raw)
        raw = reorder_channels(raw, expected_channels)

        bad_channels, channel_std = detect_bad_channels(raw, z_thresh=BAD_CHANNEL_Z_THRESH)

        row = {
            "patient_name": patient_name,
            "file_name": file_name,
            "n_channels_after_common_selection": len(raw.ch_names),
            "sampling_rate": raw.info["sfreq"],
            "duration_sec": raw.n_times / raw.info["sfreq"],
            "n_seizure_intervals": len(seizure_intervals),
            "n_bad_channels": len(bad_channels),
            "n_windows": len(y),
            "n_seizure_windows": int(np.sum(y == 1)),
            "n_nonseizure_windows": int(np.sum(y == 0))
        }

        patient_summary_rows.append(row)
        group2_all_summary_rows.append(row)
        continue

Final decision candidate:


NameError: name 'group2_global_common' is not defined